<a href="https://colab.research.google.com/github/hsmu-jeongeun/medical-data-analysis/blob/main/01_collecting_bigdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTTP Request를 활용한 데이터 수집

## Open API
- 바이오/의료 데이터가 어떻게 웹 표준 JSON으로 교환되는지 이해하기

### Ensembl API
- 전 세계적으로 가장 널리 쓰이는 유전체(Genome) 데이터베이스의 API 서버
- https://rest.ensembl.org

### URL, Query Parameter 분석
- /overlap/id/ENSG00000157764?feature=gene
- `ENSG00000157764`: 특정 유전자(Gene)를 가리키는 고유 식별자(흑색종 등 암과 연관이 깊은 BRAF 유전자 ID)
- `feature=gene`: 해당 식별자와 유전체 서열상에서 겹치는 유전자 단위의 특성 정보 요청

In [14]:
import requests, sys

# 요청 url 설정
server_url = "https://rest.ensembl.org"
ext_url = "/overlap/id/ENSG00000157764?feature=gene"

# HTTP GET method 호출
r = requests.get(server_url + ext_url, headers={ "Content-Type" : "application/json"})

# 예외 처리
if not r.ok:
  r.raise_for_status()
  sys.exit()

# Response 확인
decoded = r.json()
print(repr(decoded))

[{'id': 'ENSG00000271932', 'logic_name': 'ncrna_homo_sapiens', 'gene_id': 'ENSG00000271932', 'biotype': 'snRNA', 'description': 'RNA, U6 small nuclear 85, pseudogene [Source:HGNC Symbol;Acc:HGNC:47048]', 'start': 140884072, 'end': 140884178, 'source': 'ensembl', 'external_name': 'RNU6-85P', 'strand': 1, 'version': 1, 'seq_region_name': '7', 'assembly_name': 'GRCh38', 'feature_type': 'gene', 'canonical_transcript': 'ENST00000605989.1'}, {'assembly_name': 'GRCh38', 'strand': 1, 'version': 1, 'seq_region_name': '7', 'canonical_transcript': 'ENST00000700122.1', 'feature_type': 'gene', 'description': 'novel transcript antisense to BRAF', 'biotype': 'lncRNA', 'end': 140795643, 'start': 140767530, 'logic_name': 'havana_homo_sapiens', 'id': 'ENSG00000289788', 'gene_id': 'ENSG00000289788', 'source': 'havana'}, {'version': 15, 'seq_region_name': '7', 'strand': -1, 'assembly_name': 'GRCh38', 'feature_type': 'gene', 'canonical_transcript': 'ENST00000646891.2', 'gene_id': 'ENSG00000157764', 'logic_

### Json to Dataframe
- 반환된 JSON 데이터에는 해당 유전자의 염색체 위치(start, end, strand), 유전자 기호, 바이오타입(단백질 코딩 여부 등)과 같은 핵심 정보가 Key-Value 형태로 담겨있음
- JSON 형태의 데이터의 가독성을 높이기 위해 Dataframe 형태로 변환

In [15]:
# 데이터 프레임 형태로 저장 및 출력
import pandas as pd
df = pd.DataFrame(decoded)
df

,id,logic_name,gene_id,biotype,description,start,end,source,external_name,strand,version,seq_region_name,assembly_name,feature_type,canonical_transcript
0,ENSG00000271932,ncrna_homo_sapiens,ENSG00000271932,snRNA,"RNA, U6 small nuclear 85, pseudogene [Source:H...",140884072,140884178,ensembl,RNU6-85P,1,1,7,GRCh38,gene,ENST00000605989.1
1,ENSG00000289788,havana_homo_sapiens,ENSG00000289788,lncRNA,novel transcript antisense to BRAF,140767530,140795643,havana,NaN,1,1,7,GRCh38,gene,ENST00000700122.1
2,ENSG00000157764,ensembl_havana_gene_homo_sapiens,ENSG00000157764,protein_coding,"B-Raf proto-oncogene, serine/threonine kinase ...",140719327,140924976,ensembl_havana,BRAF,-1,15,7,GRCh38,gene,ENST00000646891.2
3,ENSG00000090266,ensembl_havana_gene_homo_sapiens,ENSG00000090266,protein_coding,NADH:ubiquinone oxidoreductase subunit B2 [Sou...,140690777,140723923,ensembl_havana,NDUFB2,1,14,7,GRCh38,gene,ENST00000247866.9


# BeautifulSoup를 활용한 크롤링

- 보건의료 관련 뉴스 기사를 수집하여 자연어 처리(NLP)분석용 말뭉치를 구축하는 과정에 활용할 수 있음
- 개발자 도구(F12)를 열어 추출이 필요한 element의 Query Selector 확인

In [16]:
!pip install beautifulsoup4

In [17]:
import requests
from bs4 import BeautifulSoup

In [18]:
# target url
url = "https://n.news.naver.com/mnews/article/015/0005265313"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36"
}

# 웹페이지 호출
response = requests.get(url, timeout=5, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

# 확인
print(soup.prettify())

<!DOCTYPE html>
<html data-useragent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36" lang="ko">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" name="viewport">
   <meta content='큐리언트 "유사 항암제 2상 성공" 소식에 이달 39% 상승' property="og:title"/>
   <meta content="article" property="og:type"/>
   <meta content="https://n.news.naver.com/mnews/article/015/0005265313" property="og:url"/>
   <meta content="https://imgnews.pstatic.net/image/015/2026/03/22/0005265313_001_20260322170915738.jpg?type=w800" property="og:image"/>
   <meta content="의약품 개발업체 큐리언트가 연구·개발(R&amp;D) 성과에 대한 기대와 상장지수펀드(ETF) 편입 효과로 투자자의 관심을 끌고 있다. 주가는 지난 20일 종가 4만4950원으로 이달 들어서만 39.4% 급등했다. 같은 기" property="og:description"/>
   <meta content="한국경제 | 네이버" property="og:article:author"/>
   <meta content="summary_large_

### 실습 문제
의료 AI 트렌드를 분석하기 위해 네이버 뉴스 기사 데이터를 수집하려고 합니다.

주어진 네이버 뉴스 기사 URL에 접속하여, 기사의 제목과 내용을 BeautifulSoup을 사용하여 추출할 수 있도록 ? 칸에 Query Selector를 입력하세요.

In [ ]:
title = soup.select_one(?).text.strip()
content = soup.select_one(?).text.strip()

In [20]:
print(f"1. 기사 제목 : {title}")
print("2. 기사 본문 (일부)")
print(f"{content[:200]}... (중략) ...")

1. 기사 제목 : 큐리언트 "유사 항암제 2상 성공" 소식에 이달 39% 상승
2. 기사 본문 (일부)
'CDK7' 억제하는 약품 기대순자산 1조 KoAct ETF두번째로 많은 6.7% 편입의약품 개발업체 큐리언트가 연구·개발(R&D) 성과에 대한 기대와 상장지수펀드(ETF) 편입 효과로 투자자의 관심을 끌고 있다. 주가는 지난 20일 종가 4만4950원으로 이달 들어서만 39.4% 급등했다. 같은 기간 코스닥지수가 2.6% 하락한 것과 비교해 눈에 띄는 상... (중략) ...
